# v13 — BiTopic Pipeline (BERTopic Baseline + Semantic–Lexical Fusion)
### Lord of the Tensors — CSE Thesis
**Multilingual Sri Lankan Hansard Speeches | 2017–2026**

## Architecture

This notebook extends the clean v12 structure with a full **BiTopic** pipeline:

```
BERTopic Baseline   BGE-M3 → UMAP-5D → HDBSCAN
                    (semantic space only)

BiTopic             BGE-M3 (α) ⊕ CountVectorizer (β)
                    → Fused Distance Matrix
                    → UMAP-5D (metric='precomputed')
                    → HDBSCAN
```

**Why BiTopic?**  
Dense semantic embeddings collapse politically distinct but semantically adjacent
terms — e.g. *"devolution"* vs *"federalism"*, or Sinhala/Tamil near-synonyms.
The lexical anchor (CountVectorizer) prevents this **embedding blur** and can
recover speeches that HDBSCAN marks as noise in the pure semantic space.

### Changes from v12
- Added BiTopic fusion pipeline (Section 6)
- Added BERTopic vs BiTopic comparison (Section 7)
- Added noise recovery analysis (Section 8)
- Extended visualisation section (Section 9)
- All v12 UMAP/HDBSCAN logic preserved and reused

---
## Section 1 — Imports & Setup

In [ ]:
# ── Suppress noisy warnings safely ───────────────────────────────────────────
# Need imported modules kept minimal to avoid notebook lint warnings.
import warnings
warnings.filterwarnings('ignore', category=UserWarning)   # UMAP precomputed metric warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Standard library ─────────────────────────────────────────────────────────
import gc
import json
import time
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'artifacts').exists():
            return candidate
    raise RuntimeError('Could not locate project root with configs/ and artifacts/.')


# ── Numeric / Data ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

# ── Clustering ────────────────────────────────────────────────────────────────
from umap import UMAP
from hdbscan import HDBSCAN

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── CPU configuration ─────────────────────────────────────────────────────────
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() or 4)
print("Imports OK ✓")

---
## Section 2 — Configuration

All tunable parameters are kept here. Adjust these before running the notebook.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION — Edit this cell before running
# ════════════════════════════════════════════════════════════════════════════════

# ── File paths ────────────────────────────────────────────────────────────────
BASE_DIR = find_project_root()
EMB_DIR    = BASE_DIR / 'embeddings'           # directory holding 20XX.npy files
DATA_CSV = BASE_DIR / 'artifacts' / 'final_v14' / 'all_speakers.csv'
SW_PATH = BASE_DIR / 'configs' / 'stopwords.json'
OUTPUT_DIR = BASE_DIR / 'artifacts' / 'v13_bitopic'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Embedding model (for reference / re-encoding if needed) ──────────────────
EMBEDDING_MODEL = 'BAAI/bge-m3'

# ── UMAP parameters ───────────────────────────────────────────────────────────
UMAP_N_NEIGHBORS  = 15
UMAP_MIN_DIST_5D  = 0.0    # tight clusters for HDBSCAN
UMAP_MIN_DIST_2D  = 0.1    # looser for visualisation
UMAP_METRIC       = 'cosine'   # for BERTopic baseline (raw embeddings)

# ── HDBSCAN parameters ────────────────────────────────────────────────────────
MIN_CLUSTER_SIZE = 10      # minimum cluster size
MIN_SAMPLES      = 5       # noise sensitivity; lower = more clusters, more noise

# ── CountVectorizer parameters ────────────────────────────────────────────────
CV_NGRAM_RANGE = (1, 2)   # unigrams + bigrams
CV_MIN_DF      = 5        # ignore terms in fewer than N speeches
# Unicode-safe token pattern — CRITICAL for Sinhala/Tamil
CV_TOKEN_PATTERN = r'(?u)[\w\u0D80-\u0DFF\u0B80-\u0BFF\u200D\u200C]+'

# ── BiTopic fusion weights ────────────────────────────────────────────────────
# α + β must equal 1.0
# α = semantic weight (BGE-M3)
# β = lexical weight  (CountVectorizer)
ALPHA = 0.85   # higher α → closer to pure BERTopic
BETA  = 0.15   # higher β → stronger keyword anchoring
assert abs(ALPHA + BETA - 1.0) < 1e-6, f'Weights must sum to 1.0, got {ALPHA+BETA}'

# ── Pipeline control flags ────────────────────────────────────────────────────
RUN_BERTOPIC  = True   # run BERTopic baseline
RUN_BITOPIC   = True   # run BiTopic pipeline

# ── Keyword extraction ────────────────────────────────────────────────────────
TOP_N_WORDS           = 50   # keywords per cluster in full output
SILHOUETTE_SAMPLE     = 5000 # max points for silhouette approximation (stratified)
LEXICAL_SVD_COMPONENTS = 300  # low-rank lexical projection size

# ── Noise recovery ────────────────────────────────────────────────────────────
RECOVERY_EXAMPLE_N = 5    # number of recovered speeches to print as examples

print(f'BASE_DIR   : {BASE_DIR}')
print(f'DATA_CSV   : {DATA_CSV}')
print(f'EMB_DIR    : {EMB_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'\nBiTopic weights: α={ALPHA} (semantic), β={BETA} (lexical)')
print(f'HDBSCAN: min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}')

---
## Section 3 — Data Loading

Load pre-computed BGE-M3 embeddings (one .npy per year) and the metadata CSV, then align them.

In [ ]:
# ── Load per-year embedding files ─────────────────────────────────────────────
year_files = sorted(
    f for f in EMB_DIR.glob('20*.npy')
    if 'master' not in f.name
)
if not year_files:
    raise FileNotFoundError(f'No 20XX.npy files found in {EMB_DIR}')

print(f'Found {len(year_files)} year files:')
year_embs = {}
for f in year_files:
    yr  = int(f.stem)
    arr = np.load(f)
    year_embs[yr] = arr
    print(f'  {f.name}: {arr.shape}')

In [ ]:
# ── Load CSV and align with embeddings ────────────────────────────────────────
df_raw = pd.read_csv(DATA_CSV)
df_raw['year'] = pd.to_datetime(df_raw['date'], errors='coerce').dt.year

print('CSV year counts:')
print(df_raw['year'].value_counts().sort_index().to_string())

print('\nPer-year alignment:')
aligned_embs = []
aligned_dfs  = []
skipped      = []
trimmed      = []

for yr in sorted(year_embs.keys()):
    emb_yr = year_embs[yr]
    df_yr  = df_raw[df_raw['year'] == yr].reset_index(drop=True)
    n_match = min(len(df_yr), len(emb_yr))

    if n_match == 0:
        skipped.append(yr)
        print(f'  [SKIP] {yr}: CSV={len(df_yr)} vs emb={len(emb_yr)} — no overlapping rows')
        continue

    if len(df_yr) != len(emb_yr):
        trimmed.append((yr, len(df_yr), len(emb_yr), n_match))
        print(f'  [WARN] {yr}: CSV={len(df_yr)} vs emb={len(emb_yr)} — using first {n_match} rows')
    else:
        print(f'  [OK]   {yr}: {n_match:>5} speeches')

    aligned_embs.append(emb_yr[:n_match])
    aligned_dfs.append(df_yr.iloc[:n_match].copy())

if not aligned_embs:
    raise RuntimeError('No yearly embeddings could be aligned with the CSV. Check the input files or the year parsing logic.')

embeddings = np.vstack(aligned_embs)
df         = pd.concat(aligned_dfs, ignore_index=True)

print(f'\nIncluded : {sorted(set(year_embs.keys()) - set(skipped))}')
print(f'Skipped  : {skipped}')
if trimmed:
    print('Trimmed  :')
    for yr, csv_n, emb_n, n_match in trimmed:
        print(f'  {yr}: CSV={csv_n}, emb={emb_n}, aligned={n_match}')
print(f'Final    : {len(df)} speeches, emb shape {embeddings.shape}')
assert len(df) == len(embeddings), 'Row count mismatch!'
print('Row counts match ✓')

# Free per-year arrays — no longer needed
del year_embs, aligned_embs, aligned_dfs, df_raw
gc.collect()
print('Memory freed ✓')

In [ ]:
# ── Dataset summary ───────────────────────────────────────────────────────────
print('=== Dataset Summary ===')
print(f'Total speeches : {len(df)}')
print(f'Columns        : {df.columns.tolist()}')
print(f'Years covered  : {sorted(df["year"].dropna().unique().astype(int).tolist())}')

# Identify text column (handles 'text', 'speech', 'speech_text', etc.)
text_col = next((c for c in ['text', 'speech', 'speech_text', 'content'] if c in df.columns), None)
if text_col is None:
    raise ValueError(f'Cannot find text column. Available: {df.columns.tolist()}')
print(f'Text column    : {text_col!r}')

# Drop rows with empty text
before = len(df)
df[text_col] = df[text_col].fillna('').astype(str)
df = df[df[text_col].str.strip().str.len() > 0].reset_index(drop=True)
embeddings = embeddings[:len(df)]   # align after drop — assumes order preserved
print(f'Dropped {before - len(df)} empty rows. Remaining: {len(df)}')

# Alias for convenience
texts = df[text_col].tolist()
years_arr = df['year'].values

print(f'\nSample speech (truncated):')
print(texts[0][:200], '...')

---
## Section 4 — Preprocessing: Trilingual Stopwords

Stopwords from v12 — Sinhala, Tamil, English — merged into a single deduplicated list.
The `CV_TOKEN_PATTERN` preserves all Unicode word characters (critical for Sinhala/Tamil).

In [ ]:
additional_stopwords = [
  "ඒ", "මේ", "ගරු", "මම", "අපි", "සහ", "ලබා", "සඳහා", "කළා", "කළ",
  "කර", "ගැන", "එතුමා", "කටයුතු", "රටේ", "අපේ", "තමයි", "නැහැ",
  "කරන්න", "කියලා", "තිබෙනවා", "එක", "අද", "වෙලා", "ඕනෑ", "තිබෙන",
  "වෙනවා", "ඒක", "ගන්න", "අය", "පත්", "පුළුවන්", "කිරීම", "දෙන්න",
  "කරන්නේ", "වලට", "වුණා", "හැබැයි", "කියන්නේ", "සියලු", "හැටියට",
  "මට", "කියන්න", "වෙන්න", "කියා", "කළේ", "වූ", "සියයට", "කියනවා",
  "වාගේ", "මේක", "අඩු", "වෙන්නේ", "කිරීමට", "වී", "කරපු", "නැති",
  "ඒවා", "මොකද", "එකක්", "කරමින්", "මගේ", "තිබුණා", "යටතේ", "යුතුයි",
  "යන්න", "එක්ක", "දන්නවා", "ඉන්න", "හිටපු", "සඳහන්", "බැහැ",
  "කාරණය", "වුණේ", "ඉතාම", "දීලා", "යුතු", "ගැනීම", "වලින්",
  "සභාවේ", "ඊට", "එකතු", "තිබුණු", "ඉල්ලා", "නැවත", "දෙන", "අදාළ",
  "අයට", "හරහා", "කරුණු", "දිහා", "බැරි", "දෙයක්", "දෙනවා", "වෙන්",
  "පස්සේ", "අවස්ථාව", "වන්නේ", "කිරීමේ", "අවසන්", "දීම", "කිසිම",
  "තමන්ගේ", "සිටින", "දේවල්", "ගිහිල්ලා", "පසු", "සම්බන්ධ",
  "එතුමාගේ", "ආරම්භ", "නේ", "විතරක්", "එකට", "බොහොම", "ඇවිල්ලා",
  "නැතිව", "පසුව", "වෙනස්", "හොඳ", "ප්‍රධාන", "තිබුණේ", "ගැනීමට",
  "වීම", "වැදගත්", "බලන්න", "ගත", "භාර", "තව", "ලද", "ගත්තා",
  "විවිධ", "වනවා", "එතුමාට", "නියෝජනය", "කාලයක්", "එවැනි", "සකස්",
  "මේවා", "ලොකු", "දා", "කරන්නට", "ඉන්නේ", "හරි", "ලබන", "අවධානය",
  "කලින්", "විශේෂ", "එකේ", "සම්බන්ධව", "ඊයේ", "ඇත්ත", "ක්‍රියා",
  "ගන්නවා", "හැකි", "විය", "ඉවත්", "එන්න", "වෙනත්", "අවස්ථාවේදී",
  "අනෙක්", "සභාවට", "ගත්", "යන්නේ", "කිව්වේ", "ඉහළ", "තත්ත්වයක්",
  "දෙන්නේ", "ඉඳලා", "තමුන්නාන්සේලා", "දේ", "එතැන", "හදනවා",
  "පුළුවන්ද", "අපිත්", "කරගන්න", "ඇතුළේ", "අපිට", "නැද්ද", "ඔය",
  "කල්", "ගත්තේ", "ගියා", "ඔවුන්ගේ", "ඇත්තටම", "කරනු", "කෙනෙක්",
  "දුන්නේ", "දීමට", "ගණනාවක්", "තරම්", "කියලායි", "අහන්න", "කියපු",
  "විතර", "ගැනීමේ", "කාරණයක්", "දන්නේ", "එන", "ගමන්", "හොඳයි",
  "නැතුව", "මොනවාද", "නතර", "නොවන", "කිසි", "දෙනා", "ආකාරයට",
  "ගිහින්", "වෙමින්", "කිරීමක්", "ඇතුළත්", "සෑම", "සිටි", "හදලා",
  "අයගේ", "විතරයි", "දිගටම", "වෙයි", "ඉදිරි", "ගැනත්", "කිහිපයක්",
  "දෙවැනි", "හිතනවා", "වශයෙන්ම", "විරුද්ධව", "තත්ත්වයට", "කඩා",
  "හිටියා", "තවම", "කටයුත්ත", "නිසි", "සියල්ල", "බල", "අවසාන",
  "ගෙවන්න", "මොන", "අවස්ථා", "දරන", "ලැබෙන", "විරුද්ධ", "ගත්ත",
  "උත්තර", "කට", "දෙක", "දෙනෙක්", "කිසිදු", "ගෙනැල්ලා", "පැත්තෙන්",
  "ඒවාට", "ඕනෑම", "හිතන", "එනවා", "පමණයි", "බලා", "බවත්", "පළමු",
  "දිය", "නිසායි", "හැකියාව", "එකඟ", "අත්සන්", "අවස්ථාවක්",
  "තවදුරටත්", "තොරව", "දෙනාම", "යුතුව", "වැඩක්", "අහනවා", "පෙනී",
  "දාලා", "ලැබුණා", "දැනුවත්", "ලැබෙනවා", "දැඩි", "ආකාරය", "දිගින්",
  "එකම", "ගැටලුවක්", "වෙනම", "සාධාරණ", "වහාම", "වුණාට", "වීමට",
  "එන්නේ", "හේතුව", "සිටියා", "සතුටු", "අයත්", "තේරුම්", "ඒකෙන්",
  "අනුමත", "ගොල්ලන්", "ඉදිරියේදී", "දවස්", "ඊළඟ", "ආවා", "අමතරව",
  "ලැබී", "කොටසක්", "වෙලාවට", "කළත්", "ක්‍රමයක්", "දැනුම්", "පැවැති",
  "දවසේ", "සහිත", "ගෙනෙන්න", "හේතු", "ආවේ", "තබා", "පිළිගන්නවා",
  "වලදී", "ඉක්මනින්", "සභාගත", "සුළු", "ගන්නට", "කළාට", "ඔක්කෝම",
  "කාරණාව", "වෙනකොට", "වුණාම", "හරියට", "ආපු", "කරන්නත්", "ලැබුණු",
  "ගියේ", "වගකීමක්", "අදාළව", "ඒකේ", "තම", "ඉන්", "ගත්තාම", "ඔබ",
  "හඳුන්වා", "ලැබුණේ", "පෙළ", "ගණනේ", "කෙටි", "පවතිනවා", "තිබෙන්න",
  "භාවිත", "ගහලා", "වේ", "නොහැකි", "සලකා", "හදන", "කැමතියි", "දීමේ",
  "වගේම", "ලැබිලා", "නැවතත්", "කවුරුත්", "හඳුනා", "පුළුවන්කම",
  "හිටියේ", "එරෙහිව", "කවදාවත්", "හේතුවෙන්", "නොමැති", "වචන",
  "කියන්නම්", "දැනගන්න", "ළඟ", "ටිකක්", "කවුරු", "මතකයි", "ඔප්පු",
  "බලාගෙන", "වතාවක්", "කළාම", "ඔක්කොම", "කරන්නම්", "අපගේ",
  "හදන්නේ", "දන්නා", "අනුගමනය", "ලැබීම", "තිබුණත්", "දමා", "හදා",
  "මුලින්ම", "දැනටමත්", "සම්පූර්ණයෙන්", "දීපු", "හෙට", "කරමු",
  "හිතන්නේ", "දමන්න", "මතය", "උපරිම", "ගනිමින්", "තහවුරු",
  "වගේ", "ම", "නෑ", "ඔව්", "ඇත", "ලබ", "ශ්‍රී", "ය", "නිසා",
  "බව", "සිට", "දී", "මහා", "මහ", "පමණ", "විට", "මෙලෙස", "ඇති",
  "සිදු", "වශයෙන්", "යන", "මගින්", "ඉතා", "එම", "අතර", "විසින්",
  "පිළිබඳව", "පිළිබඳ", "තුළ", "මෙහි", "වෙත", "වෙනුවෙන්", "වෙනුවට",
  "අනුව", "දැනට", "තවත්", "දක්වා", "මත", "ඇතුළු", "මෙසේ", "වඩා",
  "නිතර", "දැන්", "යලි", "ඉතින්", "පටන්", "තෙක්", "පවා", "වත්",
  "හැර", "මිස", "ඇයි", "හෙවත්", "නොහොත්", "ගානෙ", "බොහෝ", "සැනින්",
  "වනාහි", "අන්න", "ඔන්න", "මෙන්න", "උදෙසා", "පිණිස", "එනිසා",
  "එබැවින්", "බැවින්", "හෙයින්", "සේක", "පරිදි", "තුරු", "නමුත්",
  "වස්", "එහෙත්", "මෙය", "ලෙස", "නිසාවෙන්", "බවට", "බවෙන්",
  "මෙයින්", "ක්", "එ", "ක", "කිම", "මන්ද", "පතා", "හනික",
  "කලී", "එනමුත්", "තුලින්", "තුරා",

  "s", "us", "may", "even", "made", "say", "said", "many", "well",
  "two", "way", "go", "also", "come", "per", "one", "new", "make",
  "today", "could", "want", "think", "given", "take", "taken",
  "therefore", "cannot", "upon", "towards", "ensure", "regard",
  "members", "public", "state", "side", "believe", "high", "best",
  "past", "sure", "upon", "without", "bring", "discharge", "dignity",
  "elected", "hope", "experience", "career", "responsibility",
  "august", "assembly", "affairs", "wish", "mandates", "democracy",
  "congratulate", "behalf", "office", "party", "rights", "national",
  "political", "language", "honour", "election", "opposition",
  "rs", "cent", "dollars", "billion", "million", "sector", "bank",
  "global", "year", "years", "able", "provide", "development",
  "economy", "economic", "budget", "education", "tax", "debt",
  "foreign", "world", "international", "commission", "government",
  "president", "law", "court", "matter", "justice", "police",
  "committee", "case", "cases", "article", "supreme", "person",
  "prime", "leader", "issue", "north", "east", "human", "security",
  "resolution", "policy", "war", "important", "ministry", "digital",
  "north east", "human rights", "supreme court", "pta", "terrorism",

  "ஒரு", "என்று", "இந்த", "அவர்", "என்பது", "என", "நான்", "நாம்",
  "நீங்கள்", "அவர்கள்", "இது", "அது", "அந்த", "இல்", "க்கு", "ஆக",
  "உள்ள", "இல்லை", "மேலும்", "பற்றி", "தான்", "வந்து", "கொண்டு",
  "உள்ளது", "ஆகிய", "கௌரவ", "சபாநாயகர்", "அவர்களே", "அமைச்சர்",
  "உறுப்பினர்", "திரு", "திருமதி", "ஐயா", "பாராளுமன்ற", "மன்றம்",
  "சபை", "சபைத்தலைவர்", "மசோதா", "சட்டம்", "கேள்வி", "பதில்",
  "விவாதம்", "நேரம்", "என்ன", "எனது", "எனவும்", "அதன்", "அவ்வாறு",
  "இங்கு", "இருந்து", "இருக்கும்", "இருக்கின்றது", "உள்ளனர்",
  "உள்ளார்", "உள்ளன", "எங்கு", "எப்போது", "எப்படி", "எனவே",
  "என்பதை", "என்பதன்", "ஏன்", "ஏற்கனவே", "ஏதேனும்", "ஏனெனில்",
  "ஓர்", "ஓரு", "கொண்ட", "கொண்டே", "கொண்டும்", "கொண்டுள்ள",
  "சில", "சுமார்", "தனது", "தனி", "நமது", "நமக்கு", "நமதுடைய",
  "போது", "போல்", "மட்டும்", "மற்ற", "மற்றும்", "மற்றவை",
  "மற்றவர்கள்", "மற்றவர்", "வந்த", "வந்தது", "வந்தனர்", "வந்தான்",
  "வந்தாள்", "வந்தேன்", "தாங்கள்", "காணொளி", "தகவல்", "பலவேறு",
  "இடம்பெற்ற", "எச்சரிக்கை", "குறைந்த", "தகவல்கள்", "தொடர்பான",
  "நடவடிக்கை", "விவரம்", "காணப்படுகின்றன", "கொல்லப்பட்டனர்",
  "ஏற்பட்டது", "வழியாக", "பாதுகாப்பு", "நாடு", "செய்த", "பகுதி",
  "கொலை", "சென்று", "தற்போது", "ஊடகம்", "அதிகப்படியான", "விலை",
  "ஒன்றாக", "அமைச்சு", "வெளியில்", "செய்தி", "வெளியிடப்பட்டது",
  "விவரங்கள்", "மூலமாக", "விவரமான", "கூறப்பட்டது", "நிலைகள்",
  "செயலாளர்", "வெளியிடப்பட்டுள்ளன", "மூலம்", "கூறினார்",
  "வழங்கினார்", "செய்யப்பட்டுள்ளது", "அங்குள்ள", "அடிப்படையில்",
  "உங்களுடைய", "கேட்டுக்", "இருந்த", "செய்து", "அதாவது", "போன்ற",
  "முடியும்", "அவர்களுக்கு", "விரும்புகின்றேன்", "என்னுடைய",
  "இருக்கின்றார்கள்", "விடயம்", "தொடர்பில்", "இந்தத்", "வேண்டுமென",
  "இருக்கிறது", "நிலையில்", "அந்தக்", "பல்வேறு", "வேண்டும்",
  "நாங்கள்", "கடந்த", "இந்தச்", "அல்லது", "ஆனால்", "அரசாங்கம்",
  "அரசியல்", "அங்கு", "என்ற", "எமது", "எங்களுடைய", "தமிழ்",
  "மக்கள்", "நாட்டில்", "இன்று", "பல", "இருக்கின்ற", "இந்தப்",
  "மிகவும்", "வடக்கு", "நாட்டின்", "குறிப்பாக", "வகையில்",
  "தலைமைதாங்கும்", "ஆகவே", "முஸ்லிம்", "அந்தப்", "நன்றி",
  "மக்களின்", "மக்களுக்கு", "பிரதேச", "மிக", "ஆண்டு", "நாட்டிலே",
  "அவர்களுடைய", "ரூபாய்", "இருக்கின்றன", "பொருளாதார", "முடியாது",
  "காரணமாக", "நிதி", "சனாதிபதி", "வரவு", "மக்களுடைய", "செய்ய",
  "இன்னும்", "தேசிய", "பாரிய", "அபிவிருத்தி", "இலங்கை", "மாண்புமிகு",
  "இவ்வாறான", "கிழக்கு", "மீண்டும்", "காலத்தில்", "மாகாண",
  "கொள்கின்றேன்", "தங்களுடைய", "அரச", "ஜனாதிபதி", "இன்றைய",
  "இரண்டு", "மாவட்டத்தில்", "காணப்படுகின்றது", "மாவட்ட", "மட்டக்களப்பு",
  "புத்தளம்", "இருந்தார்", "தலைவர்", "டாக்டர்", "சிறந்த",
  "உறுப்பினராக", "அனுதாபப்", "ஒருவர்", "இருந்தது", "தேசியக்",
  "கட்சி", "அனுதாபத்தைத்", "பின்னர்", "லங்கா", "எஸ்", "அமைச்சராக",
  "தன்னுடைய", "காங்கிரஸ்", "ஸ்ரீ", "ஸ்ரீ லங்கா", "கட்சியின்",
  "மலையக", "சனாதிபதி", "திருத்தச்", "தீர்வு", "சர்வதேச", "நீதி",
  "விடயங்கள்", "சிங்கள", "ரணில்", "நாட்டை", "சபையில்", "நாட்டு",
  "நாட்டினுடைய", "பாராளுமன்றத்தில்", "பிரச்சினைகள்", "தமிழ்த்",
  "ஜனாதிபதி", "மலையக", "ரூபாய்", "பெருந்தோட்டக்", "சம்பள",
  "தோட்டத்", "பெருந்தோட்ட", "காணி", "இருக்கின்ற", "கம்பனிகள்",
  "சம்பளம்", "மக்களை", "தொழில்", "பெருந்தோட்டத்", "உயர்வு",
  "கிட்டத்தட்ட", "சம்பள உயர்வு", "தொழிலாளர்களுக்கு", "பிரச்சினை",
  "ஆயிரம்", "தொழிலாளர்கள்", "வேலை", "தோட்ட", "200",
  "தொழிலாளர்களின்", "சபையிலே", "இன்றைக்கு", "பெருந்தோட்டக் கம்பனிகள்",
  "பெருந்தோட்டப்", "1 000", "பல", "அந்தப்", "நிதி", "பிரதேச",
  "அபிவிருத்தி", "குறிப்பாக", "இருக்கின்றன", "கொள்கின்றேன்",
  "மிகவும்", "விளையாட்டு", "போக்குவரத்து", "கேட்கின்றேன்",
  "மாவட்டத்தில்", "வீதி", "போக்குவரத்துச்", "புகையிரத", "இன்னும்",
  "கிட்டத்தட்ட", "காணப்படுகின்றது", "பாடசாலை", "மேலதிக", "மில்லியன்",
  "நிலை", "வட", "சேவை", "பாலம்", "வீதிகள்", "காலங்களில்", "மன்னார்",
  "கிளிநொச்சி", "சேவையை", "மாகாண", "பிரதேசத்தில்",
  "hospital", "medical", "வைத்தியசாலை", "வைத்தியசாலையில்",
  "வைத்திய", "வைத்தியசாலைகள்", "வைத்தியர்கள்", "சுகாதார",
  "hospitals", "வைத்தியசாலைக்கு", "முல்லைத்தீவு", "province",
  "district", "ஆதார", "ஏனைய", "பொது", "officers", "எடுக்க",
  "வவுனியா", "உங்களுக்கு", "இப்போது", "general", "ct",
  "medical officers", "மருத்துவ", "முடியுமா", "base",
  "காணி", "காணிகள்", "காணிகளை", "மீனவர்கள்", "ஏக்கர்",
  "பிரதேசத்தில்", "முல்லைத்தீவு", "வவுனியா", "இந்திய",
  "கடற்றொழில்", "விமான", "நீர்", "கேட்டுக்கொள்கின்றேன்",
  "திணைக்களம்", "அதனை",

  "hon", "speaker", "member", "parliament", "sir", "madam", "minister",
  "honorable", "house", "question", "answer", "debate", "bill", "act",
  "amendment", "point", "order", "chair", "time", "minute", "gentleman",
  "friend", "yes", "no", "hear", "parliamentary", "thank", "country",
  "going", "report", "like", "model", "extra", "much", "must",
  "project", "system", "come", "per", "also", "one", "mr",
  "the", "and", "is", "in", "to", "of", "it", "that", "this", "for",
  "on", "with", "as", "by", "at", "be", "are", "was", "were", "from",
  "have", "has", "had", "not", "but", "or", "an", "they", "which",
  "you", "we", "i", "my", "me", "their", "will", "would", "what",
  "who", "whom", "these", "those", "am", "been", "being", "do", "does",
  "did", "doing", "a", "if", "because", "until", "while", "about",
  "against", "between", "into", "through", "during", "before", "after",
  "above", "below", "up", "down", "out", "off", "over", "under",
  "again", "further", "then", "once", "here", "there", "when", "where",
  "why", "how", "all", "any", "both", "each", "few", "more", "most",
  "other", "some", "such", "nor", "only", "own", "same", "so", "than",
  "too", "very", "can", "just", "should", "now", "himself", "herself",
  "itself", "themselves", "ourselves", "yourself", "yourselves",
  "myself", "ours", "yours", "hers", "theirs", "him", "her", "its",
  "he", "she", "s"
]

In [ ]:
new_stops_to_add = [
    "තිබෙනවා", "කියලා", "කරන්න", "නැහැ", "තමයි", "අපේ", "ලබා", "එක", "කටයුතු", 
    "අද", "වෙලා", "ඕනෑ", "තිබෙන", "අපට", "කියන", "කළා", "වෙනවා", "ඒක", "ගන්න", 
    "අය", "පත්", "පුළුවන්", "කළ", "තිබෙන්නේ", "අවශ්‍ය", "ඉදිරිපත්", "කිරීම", "එතුමා", 
    "වාගේම", "දෙන්න", "වැඩ", "මා", "නොවෙයි", "විධියට", "කරන්නේ", "වලට", "සම්බන්ධයෙන්", 
    "වුණා", "රටේ", "කථානායකතුමනි", "කථා", "රට", "කොට", "හැබැයි", "ක්‍රියාත්මක", "රාජ්‍ය", 
    "කියන්නේ", "විශාල",
    "තිකබනවා", "නැා", "ත්මයි", "රානන", "රාා්", "කම්", "කියනා", "තිකබන්කන්", "තිකබන",
    "සියලු", "හැටියට", "මට", "යම්", "කියන්න", "වෙන්න", "කියා", "කළේ", "වූ", "සියයට", 
    "මන්ත්‍රීතුමනි", "කියනවා", "වාගේ", "මේක", "අඩු", "වෙන්නේ", "කිරීමට", "වී", "කරපු", 
    "නැති", "ඒවා", "මොකද", "එකක්", "අවුරුදු", "කරමින්", "මගේ", "තිබුණා", "යටතේ", 
    "මූලාසනාරූඪ", "විශේෂයෙන්ම", "යුතුයි", "යන්න", "එක්ක", "දන්නවා", "ඉන්න", "හිටපු", 
    "සඳහන්", "බැහැ", "පසුගිය", "කාරණය", "වුණේ", "ඉතාම", "දීලා", "කාලයේ", "යුතු", 
    "අරගෙන", "යොමු", "මොකක්ද", "කරගෙන", "එදා", "නැත්නම්", "කාරක", "සමහර", "ටික", 
    "සිටිනවා", "හැම", "තත්ත්වය", "කැමැතියි", "ඇමතිතුමනි", "ගිය", "තුමනි", "බලාපොරොත්තු", 
    "වුණු", "ගැනීම", "වලින්", "සභාවේ", "කථානායක", "ඊට", "එකතු", "තිබුණු", "ප්‍රකාශ", 
    "ඉල්ලා", "නැවත", "සිද්ධ", "දෙන", "අදාළ", "අයට", "හරහා", "කරුණු", "දින", "බැරි", 
    "දෙයක්", "සභාපතිතුමනි", "දෙනවා", "වෙන්", "පස්සේ", "අවස්ථාව", "වන්නේ", "කිරීමේ", 
    "ඔබතුමාට", "අවසන්", "දුන්නා", "දීම", "කිසිම", "කාලය", "තමන්ගේ", "සිටින", "ඔබතුමන්ලා", 
    "දේවල්", "ගිහිල්ලා", "පසු", "එපා", "සභාව", "සම්බන්ධ", "එතුමාගේ", "එහි", "ආරම්භ", 
    "තුළින්", "නේ", "විතරක්", "සභා", "එකට", "බොහොම", "ඇවිල්ලා", "නැතිව", "පසුව", "මඟින්", 
    "වෙනස්", "හොඳ", "ප්‍රධාන", "තිබුණේ", "ගැනීමට", "වීම", "වැදගත්", "බලන්න", "ප්‍රමාණයක්", 
    "කොහොමද", "ගත", "භාර", "තව", "ලද", "ගත්තා", "විවිධ", "වනවා", "එතුමාට", "නියෝජනය", 
    "කාලයක්", "එවැනි", "වෙලාවේ", "මුහුණ", "සකස්", "මේවා", "මීට", "ලොකු", "දා", "කරන්නට", 
    "ඉන්නේ", "හරි", "ලබන", "අවධානය", "කලින්", "විශේෂ", "එකේ", "සම්බන්ධව", "අවස්ථාවේ", 
    "ඊයේ", "ඇත්ත", "මාස", "ක්‍රියා", "කාරණා", "ගන්නවා", "හැකි", "විය", "අලුත්", "ඉවත්", 
    "එන්න", "වෙනත්", "අවස්ථාවේදී", "අනෙක්", "මෙය", "සභාවට", "ගත්", "යන්නේ", "කිව්වේ", 
    "ඉහළ", "තත්ත්වයක්", "දෙන්නේ", "ඉඳලා", "තමුන්නාන්සේලා", "දේ", "එතැන", "මේකයි", "දිහා", 
    "හදනවා", "පුළුවන්ද", "අපිත්", "කරගන්න", "යි", "ඇතුළේ", "අපිට", "නැද්ද", "ඔය", "කල්", 
    "ගත්තේ", "ගියා", "ඔවුන්ගේ", "ඇත්තටම", "කරනු", "කෙනෙක්", "දුන්නේ", "දීමට", "ගණනාවක්", 
    "උත්සාහ", "තිඛෙනවා", "නිකුත්", "තරම්", "මැදිහත්", "කියලායි", "අහන්න", "කියපු", 
    "තිබෙනවාද", "විතර", "එසේ", "ගැනීමේ", "කාරණයක්", "දන්නේ", "එන", "අවශ්‍යයි", "ගමන්", 
    "හොඳයි", "නැතුව", "මොනවාද", "තිස්සේ", "නතර", "නොවන", "කිසි", "දෙනා", "ආකාරයට", 
    "පළ", "ගිහින්", "වෙමින්", "කාල", "කිරීමක්", "ඇතුළත්", "සෑම", "සිටි", "ගත්තොත්", 
    "හදලා", "අත්", "අයගේ", "විතරයි", "දැන", "දිගටම", "වෙයි", "ඉදිරි", "යම්කිසි", "ගැනත්", 
    "කිහිපයක්", "දෙවැනි", "හිතනවා", "වශයෙන්ම", "විරුද්ධව", "තත්ත්වයට", "මෙතැන", "කඩා", 
    "හිටියා", "තවම", "කටයුත්ත", "නිසි", "දීර්ඝ", "මෙයයි", "සියල්ල", "බල", "අවසාන", 
    "ගෙවන්න", "අහිමි", "මොන", "අවස්ථා", "දරන", "ලැබෙන", "ඉෂ්ට", "විරුද්ධ", "ගත්ත", 
    "අවුරුද්දේ", "උත්තර", "කට", "දෙක", "දෙනෙක්", "කිසිදු", "නැද්ද", "ගෙනැල්ලා", 
    "පැත්තෙන්", "ඒවාට", "ඕනෑම", "පළමුවැනි", "හිතන", "එනවා", "පමණයි", "බලා", "බවත්", 
    "පළමු", "දිය", "නිසායි", "හැකියාව", "එකඟ", "එල්ල", "අත්සන්", "මේකට", "අවස්ථාවක්", 
    "තවදුරටත්", "තොරව", "දෙනාම", "යුතුව", "වැඩක්", "අහනවා", "පෙනී", "දාලා", "ලැබුණා", 
    "කළොත්", "පිට", "දැනුවත්", "ලැබෙනවා", "දැඩි", "ආකාරය", "සූදානම්", "දිගින්", "අවම", 
    "එකම", "ගැටලුවක්", "වෙනම", "මතක", "සාධාරණ", "වහාම", "වුණාට", "වීමට", "එන්නේ", 
    "තීන්දුවක්", "හේතුව", "එළියට", "සිටියා", "සතුටු", "අයත්", "තේරුම්", "මෙහෙම", 
    "පිළිබඳවත්", "ඒකෙන්", "අනුමත", "වුණොත්", "ගොල්ලන්", "එකත්", "ඉදිරියේදී", "අර", 
    "එකෙන්", "දවස්", "ඊළඟ", "ආවා", "අමතරව", "වනකොට", "ඒකයි", "යහ", "ලැබී", "ආසන්න", 
    "කොටසක්", "වෙලාවට", "කළත්", "ක්‍රමයක්", "ඔබතුමන්ලාට", "දැනුම්", "පැවැති", "one", 
    "පොඩි", "දවසේ", "සහිත", "ගෙනෙන්න", "හේතු", "ආවේ", "තබා", "කිරීමේදී", "පිළිගන්නවා", 
    "වලදී", "ඉක්මනින්", "සභාගත", "සුළු", "ගන්නට", "කළාට", "එතුමන්ලා", "සම්පූර්ණයෙන්ම", 
    "must", "යුත්තේ", "දිහා", "ගනිමින්", "තහවුරු", "ඔක්කෝම", "කාරණාව", "වෙනකොට", 
    "වුණාම", "හරියට", "ආපු", "කරන්නත්", "ලැබුණු", "අනිවාර්යයෙන්ම", "ගියේ", "වගකීමක්", 
    "අදාළව", "ඒකේ", "තම", "හම්බ", "ඉන්", "තුනක්", "ගත්තාම", "ඔබ", "හඳුන්වා", "ලැබුණේ", 
    "සාමාන්‍යයෙන්", "පෙළ", "ගණනේ", "කෙටි", "පවතිනවා", "තිබෙන්න", "භාවිත", "ගහලා", 
    "වේ", "නොහැකි", "සලකා", "හදන", "කැමතියි", "දීමේ", "වගේම", "ලැබිලා", "නැවතත්", 
    "කවුරුත්", "හඳුනා", "පුළුවන්කම", "හිටියේ", "එරෙහිව", "එයට", "කවදාවත්", "හේතුවෙන්", 
    "නොමැති", "වචන", "කියන්නම්", "දැනගන්න", "පැත්තකින්", "ළඟ", "ටිකක්", "කවුරු", 
    "අයින්", "මතකයි", "ඔප්පු", "අහන්නේ", "පැය", "බලාගෙන", "වතාවක්", "කළාම", "ඔක්කොම", 
    "කරන්නම්", "අපගේ", "හදන්නේ", "දන්නා", "අනුගමනය", "මොකක්", "ලැබීම", "තිබුණත්", 
    "දමා", "කෙරෙහි", "හදා", "මුලින්ම", "උදාහරණයක්", "තීරණයක්", "දැනටමත්", "සම්පූර්ණයෙන්", 
    "දීපු", "හෙට", "කරමු", "හිතන්නේ", "දමන්න", "එවකට", "මතය", "උපරිම",
    "வேண்டும்", "நாங்கள்", "கடந்த", "இந்தச்", "அல்லது", "ஆனால்", "அரசாங்கம்", "அரசியல்", 
    "அங்கு", "என்ற", "எமது", "எங்களுடைய", "தமிழ்", "மக்கள்", "நாட்டில்", "இன்று", "also", "come", "per", "system", "project"
]

In [ ]:
# ── Tier 1: Sinhala ──────────────────────────────────────────────────────────
sinhala_stops = [
    "නව", "මන", "අප", "කරන", "වන", "සහ", "හා", "මෙම", "ඒ", "බව", "මේ", "එක්",
    "සඳහා", "මම", "අපි", "ඔහු", "ඇය", "ඔවුන්", "එය", "ද", "ට", "ගේ", "වල", "නම්",
    "එතකොට", "ඒකට", "එහෙම", "මෙ", "ඒකෙ", "කිව්වා", "කරලා", "තියෙනවා", "කරනවා",
    "යනවා", "ඉන්නවා", "ගෙන", "ගැන", "වගේ", "ම", "ද්", "නෑ", "ඔව්", "ඇත", "ලබ", "කර",
    "ගරු", "මහතා", "මන්ත්‍රීතුමා", "තුමා", "කතානායකතුමනි", "ඔබතුමා", "කථානායකතුමා",
    "අමාත්‍යතුමා", "ඇමතිතුමා", "මන්ත්‍රීවරයා", "නියෝජ්‍ය", "පාර්ලිමේන්තුව", "සභාපතිතුමා",
    "මහත්මයා", "මහත්මිය", "යෝජනාව", "පනත", "පිළිතුර", "ප්‍රශ්නය", "විවාදය", "විනාඩි", "කාලය"
    "සහ","සමග","සමඟ","අහා","ආහ්","ආ","ඕහෝ","අනේ","අඳෝ","අපොයි","අපෝ","අයියෝ","ආයි","ඌයි",
    "චී","චිහ්","චික්","හෝ","දෝ","දෝහෝ","මෙන්","සේ","වැනි","බඳු","වන්","අයුරු","අයුරින්",
    "ලෙස","වැඩි","ශ්‍රී","හා","ය","නිසා","නිසාවෙන්","බවට","බව","බවෙන්","නම්","සිට","දී",
    "මහා","මහ","පමණ","පමණින්","වන","විට","මේ","මෙලෙස","මෙයින්","ඇති","සිදු","වශයෙන්",
    "යන","සඳහා","මගින්","ඉතා","ඒ","එම","ද","අතර","විසින්","පිළිබඳව","පිළිබඳ","තුළ",
    "මෙම","මෙහි","වෙත","වෙතින්","වෙතට","වෙනුවෙන්","වෙනුවට","වෙන","ගැන","අනුව","දැනට",
    "තවත්","දක්වා","ට","ගේ","එ","ක","ක්","මත","ඇතුළු","මෙසේ","වඩා","වඩාත්ම","නිති",
    "නිතර","දැන්","යලි","පුන","ඉතින්","පටන්","තෙක්","පවා","වත්","විනා","හැර","මිස",
    "කිම","ඇයි","මන්ද","හෙවත්","නොහොත්","පතා","ගානෙ","බොහෝ","වහා","සැනින්","හනික",
    "නම්","වනාහි","කලී","අන්න","ඔන්න","මෙන්න","උදෙසා","පිණිස","අරබයා","එනිසා",
    "එබැවින්","බැවින්","හෙයින්","සේක","පරිදි","තුරු","තුරා","තුලින්","නමුත්","එනමුත්",
    "වස්","එහෙත්"
]
tamil_stops = [
    "மற்றும்", "ஒரு", "என்று", "இந்த", "அவர்", "என்பது", "என", "நான்", "நாம்", "நீங்கள்",
    "அவர்கள்", "இது", "அது", "அந்த", "இல்", "க்கு", "ஆக", "உள்ள", "இல்லை", "மேலும்",
    "பற்றி", "தான்", "வந்து", "கொண்டு", "உள்ளது", "ஆகிய",
    "கௌரவ", "சபாநாயகர்", "அவர்களே", "அமைச்சர்", "உறுப்பினர்", "திரு", "திருமதி", "ஐயா",
    "பாராளுமன்ற", "மன்றம்", "சபை", "சபைத்தலைவர்", "மசோதா", "சட்டம்", "கேள்வி", "பதில்",
    "விவாதம்", "நேரம்",
    "ஒரு","என்று","இந்த","என","இது","அது","என்ன","எனது","எனவும்",
    "அதன்","அவர்","அவர்கள்","அவ்வாறு","இங்கு","இருந்து","இருக்கும்",
    "இருக்கின்றது","உள்ள","உள்ளது","உள்ளனர்","உள்ளார்","உள்ளன",
    "எங்கு","எப்போது","எப்படி","எனவே","என்பது","என்பதை","என்பதன்",
    "ஏன்","ஏற்கனவே","ஏதேனும்","ஏனெனில்","ஓர்","ஓரு","கொண்ட","கொண்டு",
    "கொண்டே","கொண்டும்","கொண்டுள்ள","சில","சுமார்","தான்","தான்",
    "தனது","தனது","தனி","தான்","நான்","நாம்","நீங்கள்","நீங்கள்",
    "நீ","நான்","நாம்","நமது","நமக்கு","நமதுடைய","நீங்கள்","நீங்கள்",
    "போது","போல்","மட்டும்","மற்ற","மற்றும்","மற்றவை","மற்றவர்கள்",
    "மற்றவர்","மற்றும்","மற்றவை","மற்றும்","மற்றவை","மற்றும்",
    "வந்து","வந்த","வந்தது","வந்தனர்","வந்தான்","வந்தாள்","வந்தது",
    "வந்தனர்","வந்தேன்","வந்து","வந்து","வந்து","வந்து","வந்தனர்",
    "வந்தது","வந்தது","வந்தது","வந்தது","வந்தனர்","வந்தனர்",
    "தாங்கள்","காணொளி","தகவல்","பலவேறு","இடம்பெற்ற","எச்சரிக்கை",
    "குறைந்த","வந்த","தகவல்கள்","தொடர்பான","நடவடிக்கை","விவரம்",
    "காணப்படுகின்றன","கொல்லப்பட்டனர்","ஏற்பட்டது","வழியாக","பாதுகாப்பு",
    "நாடு","செய்த","பகுதி","கொலை","சென்று","தற்போது","ஊடகம்","உள்ளது",
    "அதிகப்படியான","விலை","ஒன்றாக","அமைச்சு","வெளியில்","செய்தி",
    "வெளியிடப்பட்டது","விவரங்கள்","நடவடிக்கை","மூலமாக","விவரமான",
    "கூறப்பட்டது","நிலைகள்","செயலாளர்","விவரங்கள்","வெளியிடப்பட்டுள்ளன",
    "மூலம்","பற்றி","கூறினார்","அமைச்சர்","வழங்கினார்","கூறினார்",
    "செய்யப்பட்டுள்ளது","கூறினார்",
    "அங்குள்ள", "அடிப்படையில்", "உங்களுடைய", "கேட்டுக்", "இருந்த", 
    "செய்து", "அதாவது", "போன்ற", "முடியும்", "அவர்களுக்கு", 
    "விரும்புகின்றேன்", "என்னுடைய", "இருக்கின்றார்கள்", "விடயம்", 
    "தொடர்பில்", "இந்தத்", "வேண்டுமென", "இருக்கிறது", "நிலையில்", 
    "அந்தக்", "பல்வேறு"
]
english_stops = [
    "the", "and", "is", "in", "to", "of", "it", "that", "this", "for", "on", "with",
    "as", "by", "at", "be", "are", "was", "were", "from", "have", "has", "had", "not",
    "but", "or", "an", "they", "which", "you", "we", "i", "my", "me", "their", "will", "would",
    "mr", "hon", "speaker", "member", "parliament", "sir", "madam", "minister",
    "honorable", "house", "question", "answer", "debate", "bill", "act", "amendment",
    "point", "order", "chair", "time", "minute", "gentleman", "friend", "yes", "no", "hear",
    "i","me","my","myself","we","our","ours","ourselves","you","your","yours",
    "yourself","yourselves","he","him","his","himself","she","her","hers",
    "herself","it","its","itself","they","them","their","theirs","themselves",
    "what","which","who","whom","this","that","these","those","am","is","are",
    "was","were","be","been","being","have","has","had","having","do","does",
    "did","doing","a","an","the","and","but","if","or","because","as","until",
    "while","of","at","by","for","with","about","against","between","into",
    "through","during","before","after","above","below","to","from","up","down",
    "in","out","on","off","over","under","again","further","then","once","here",
    "there","when","where","why","how","all","any","both","each","few","more",
    "most","other","some","such","no","nor","not","only","own","same","so",
    "than","too","very","can","will","just","should","now",
    "come", "per", "system", "project", "thank", "country", 
    "going", "report", "like", "model", "extra", "much", 
    "parliamentary"
]

CUSTOM_STOPWORDS = list(set(sinhala_stops + tamil_stops + english_stops + new_stops_to_add + additional_stopwords))
print(f"Total stopwords: {len(CUSTOM_STOPWORDS)}")
print(f"Total stopwords loaded : {len(CUSTOM_STOPWORDS)}")
print(f"  Sinhala : {len(sinhala_stops)}")
print(f"  Tamil   : {len(tamil_stops)}")
print(f"  English : {len(english_stops)}")


---
## Section 5 — BERTopic Baseline Pipeline

Standard pipeline: BGE-M3 embeddings → UMAP-5D → HDBSCAN.  
UMAP projections are cached to disk so the expensive step only runs once.

In [ ]:
if not RUN_BERTOPIC:
    print('RUN_BERTOPIC=False — skipping BERTopic baseline.')
else:
    # ── UMAP 5D: for clustering ───────────────────────────────────────────────
    bert_umap5_path = OUTPUT_DIR / 'bert_umap5.npy'
    bert_umap2_path = OUTPUT_DIR / 'bert_umap2.npy'

    if bert_umap5_path.exists() and bert_umap2_path.exists():
        print('Loading cached BERTopic UMAP projections ...')
        bert_emb5 = np.load(bert_umap5_path)
        bert_emb2 = np.load(bert_umap2_path)
        assert len(bert_emb5) == len(embeddings), \
            f'Cached UMAP5 rows ({len(bert_emb5)}) != embeddings ({len(embeddings)}). Delete cache and re-run.'
        print(f'  emb5: {bert_emb5.shape},  emb2: {bert_emb2.shape}')
    else:
        print(f'Fitting BERTopic UMAP 5D (n_neighbors={UMAP_N_NEIGHBORS}) ...')
        t0 = time.time()
        umap5 = UMAP(
            n_neighbors  = UMAP_N_NEIGHBORS,
            n_components = 5,
            min_dist     = UMAP_MIN_DIST_5D,
            metric       = UMAP_METRIC,
            random_state = RANDOM_STATE
        )
        bert_emb5 = umap5.fit_transform(embeddings)
        np.save(bert_umap5_path, bert_emb5)
        print(f'  Done in {time.time()-t0:.1f}s — shape: {bert_emb5.shape}')

        print('Fitting BERTopic UMAP 2D (for visualisation) ...')
        t0 = time.time()
        umap2 = UMAP(
            n_neighbors  = UMAP_N_NEIGHBORS,
            n_components = 2,
            min_dist     = UMAP_MIN_DIST_2D,
            metric       = UMAP_METRIC,
            random_state = RANDOM_STATE
        )
        bert_emb2 = umap2.fit_transform(embeddings)
        np.save(bert_umap2_path, bert_emb2)
        print(f'  Done in {time.time()-t0:.1f}s — shape: {bert_emb2.shape}')

In [ ]:
if not RUN_BERTOPIC:
    print('RUN_BERTOPIC=False — skipping.')
else:
    print(f'Running BERTopic HDBSCAN (min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}) ...')
    hdb_bert = HDBSCAN(
        min_cluster_size      = MIN_CLUSTER_SIZE,
        min_samples           = MIN_SAMPLES,
        metric                = 'euclidean',
        cluster_selection_method = 'eom',
        prediction_data       = True,
    )
    bert_labels = hdb_bert.fit_predict(bert_emb5)

    K_bert      = len(set(bert_labels)) - (1 if -1 in bert_labels else 0)
    noise_bert  = (bert_labels == -1).sum()
    noise_pct_b = 100 * noise_bert / len(bert_labels)

    print(f'\n── BERTopic Baseline Results ──────────────────────────────')
    print(f'  Speeches       : {len(bert_labels)}')
    print(f'  Topics found   : {K_bert}')
    print(f'  Noise (-1)     : {noise_bert} ({noise_pct_b:.1f}%)')
    print(f'  Clustered      : {len(bert_labels) - noise_bert} ({100-noise_pct_b:.1f}%)')

---
## Section 6 — BiTopic Pipeline

### 6.1 Lexical Space — CountVectorizer

The lexical matrix captures **exact keyword overlap** between speeches. Speeches
that share distinctive terms (e.g. *IMF*, *devolution*, *HRCSL*) will have high
lexical similarity even if their dense embeddings diverge.

Key design choices:
- `token_pattern` preserves Sinhala (U+0D80–U+0DFF) and Tamil (U+0B80–U+0BFF) Unicode
- `min_df=5` filters rare noise terms
- L2 normalisation ensures speech length does not bias similarity

In [ ]:
if not RUN_BITOPIC:
    print('RUN_BITOPIC=False — skipping BiTopic pipeline.')
else:
    print('Fitting CountVectorizer (lexical space) ...')
    vectorizer = CountVectorizer(
        stop_words    = CUSTOM_STOPWORDS,
        ngram_range   = CV_NGRAM_RANGE,
        min_df        = CV_MIN_DF,
        token_pattern = CV_TOKEN_PATTERN,
    )
    lex_matrix_raw = vectorizer.fit_transform(texts)  # sparse (N × vocab)
    vocab_size     = len(vectorizer.vocabulary_)
    N              = len(texts)

    print(f'Vocabulary size      : {vocab_size} terms')
    print(f'Lexical matrix shape : {lex_matrix_raw.shape}')
    print(f'Sparsity             : {100*(1 - lex_matrix_raw.nnz/(N*vocab_size)):.1f}%')

    # Print top terms for a sanity check
    feature_names = vectorizer.get_feature_names_out()
    term_counts   = np.asarray(lex_matrix_raw.sum(axis=0)).flatten()
    top_idx       = term_counts.argsort()[::-1][:15]
    print('\nTop 15 terms by corpus frequency:')
    for i in top_idx:
        print(f'  {feature_names[i]:<35} {int(term_counts[i]):>6} occurrences')

### 6.2 Semantic Space — L2-Normalise BGE-M3 Embeddings

We L2-normalise the BGE-M3 embeddings so that `embeddings @ embeddings.T`
gives cosine similarities directly, matching the scale of the lexical similarity matrix.

In [ ]:
if not RUN_BITOPIC:
    print('RUN_BITOPIC=False — skipping.')
else:
    # L2-normalise embeddings for cosine similarity via dot product
    emb_norm = normalize(embeddings, norm='l2').astype(np.float32)

    # L2-normalise lexical matrix rows before low-rank projection
    lex_matrix_norm = normalize(lex_matrix_raw, norm='l2')

    n_lex_components = min(
        LEXICAL_SVD_COMPONENTS,
        max(2, min(lex_matrix_norm.shape[0] - 1, lex_matrix_norm.shape[1] - 1)),
)
    print(f'Projecting lexical space with TruncatedSVD(n_components={n_lex_components}) ...')
    t0 = time.time()
    lex_svd = TruncatedSVD(n_components=n_lex_components, random_state=RANDOM_STATE)
    lex_latent = lex_svd.fit_transform(lex_matrix_norm)
    lex_latent = normalize(lex_latent, norm='l2').astype(np.float32)

    lexical_var = float(lex_svd.explained_variance_ratio_.sum())
    print(f'  Done in {time.time()-t0:.1f}s | shape: {lex_latent.shape}')
    print(f'  Lexical explained variance ratio : {lexical_var:.4f}')
    print(f'  emb_norm shape                    : {emb_norm.shape}  dtype={emb_norm.dtype}')
    print(f'  lex_latent shape                  : {lex_latent.shape}  dtype={lex_latent.dtype}')
    print('  Next cell will combine semantic and lexical features for UMAP.')

### 6.3 Feature Fusion — The BiTopic Formula

Instead of materializing a full pairwise similarity matrix, this notebook keeps the
semantic and lexical views as compact feature vectors and fuses them into one
combined representation before UMAP.

$$X_{\text{final}} = [\alpha \cdot X_{\text{semantic}} \;\Vert\; \beta \cdot X_{\text{lexical}}]$$

**Why this design?**  
BGE-M3 preserves multilingual semantic alignment, while the lexical projection keeps
keyword-specific structure without allocating an $N \times N$ distance matrix.

In [ ]:
if not RUN_BITOPIC:
    print('RUN_BITOPIC=False — skipping.')
else:
    print(f'Combining semantic and lexical features: α={ALPHA}, β={BETA}')

    bitopic_features = np.hstack([
        ALPHA * emb_norm,
        BETA * lex_latent,
    ]).astype(np.float32, copy=False)
    bitopic_features = normalize(bitopic_features, norm='l2').astype(np.float32)

    print(f'  Combined feature shape : {bitopic_features.shape}')
    print('  UMAP will run directly on the combined feature space.')

    # Free intermediate matrices before the expensive UMAP step.
    del lex_matrix_norm, emb_norm   # lex_latent kept for late-fusion pipeline
    gc.collect()
    print('Intermediate matrices freed ✓')

### 6.4 BiTopic UMAP + HDBSCAN

We pass the combined feature matrix into UMAP directly, so the notebook avoids
both the dense $N \times N$ similarity matrix and the on-disk memmap workaround.

In [ ]:
if not RUN_BITOPIC:
    print('RUN_BITOPIC=False — skipping.')
else:
    # ── UMAP 5D (for clustering) ──────────────────────────────────────────────
    bi_umap5_path = OUTPUT_DIR / 'bitopic_umap5.npy'
    bi_umap2_path = OUTPUT_DIR / 'bitopic_umap2.npy'

    if bi_umap5_path.exists() and bi_umap2_path.exists():
        print('Loading cached BiTopic UMAP projections ...')
        bi_emb5 = np.load(bi_umap5_path)
        bi_emb2 = np.load(bi_umap2_path)
        assert len(bi_emb5) == N, \
            f'Cached BiTopic UMAP5 rows ({len(bi_emb5)}) != N ({N}). Delete cache and re-run.'
        print(f'  emb5: {bi_emb5.shape},  emb2: {bi_emb2.shape}')
    else:
        print(f'Fitting BiTopic UMAP 5D on combined feature space ...')
        t0 = time.time()
        umap_bi5 = UMAP(
            n_components = 5,
            n_neighbors  = UMAP_N_NEIGHBORS,
            min_dist     = UMAP_MIN_DIST_5D,
            metric       = 'cosine',
            random_state = RANDOM_STATE
        )
        bi_emb5 = umap_bi5.fit_transform(bitopic_features)
        np.save(bi_umap5_path, bi_emb5)
        print(f'  Done in {time.time()-t0:.1f}s | shape: {bi_emb5.shape}')

        print('Fitting BiTopic UMAP 2D for visualisation ...')
        t0 = time.time()
        umap_bi2 = UMAP(
            n_components = 2,
            n_neighbors  = UMAP_N_NEIGHBORS,
            min_dist     = UMAP_MIN_DIST_2D,
            metric       = 'cosine',
            random_state = RANDOM_STATE
        )
        bi_emb2 = umap_bi2.fit_transform(bitopic_features)
        np.save(bi_umap2_path, bi_emb2)
        print(f'  Done in {time.time()-t0:.1f}s | shape: {bi_emb2.shape}')

    # Free the combined feature matrix now that UMAP is done
    if 'bitopic_features' in dir():
        del bitopic_features
        gc.collect()
        print('bitopic_features freed ✓')

In [ ]:
if not RUN_BITOPIC:
    print('RUN_BITOPIC=False — skipping.')
else:
    print(f'Running BiTopic HDBSCAN (min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}) ...')
    hdb_bi = HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
    )
    bi_labels = hdb_bi.fit_predict(bi_emb5)

    K_bi = len(set(bi_labels)) - (1 if -1 in bi_labels else 0)
    noise_bi = (bi_labels == -1).sum()
    noise_pct_bi = 100 * noise_bi / len(bi_labels)

    print(f'\n── BiTopic Results (α={ALPHA}, β={BETA}) ──────────────────────')
    print(f'  Speeches       : {len(bi_labels)}')
    print(f'  Topics found   : {K_bi}')
    print(f'  Noise (-1)     : {noise_bi} ({noise_pct_bi:.1f}%)')
    print(f'  Clustered      : {len(bi_labels) - noise_bi} ({100-noise_pct_bi:.1f}%)')

---
## Section 6.5 — Late Fusion Pipeline (Alternative BiTopic)

**Why late fusion works better:**
The early-fusion approach (cell 6.3) feeds a 1152-dimensional mixed matrix into UMAP,
forcing UMAP to simultaneously resolve semantic *and* lexical geometry.
Late fusion lets each modality find its own low-dimensional structure first,
then merges the two clean 5-D projections — giving HDBSCAN a 10-D space where
both semantic proximity and keyword overlap contribute directly.

Architecture:
```
BGE-M3 embeddings (1024-D)  →  UMAP-5D  →  bert_emb5  ─┐
                                                          ├─ α·bert + β·lex → HDBSCAN
lex_latent (128-D SVD)      →  UMAP-5D  →  lex_emb5   ─┘
```


In [ ]:
if not RUN_BITOPIC:
    print('RUN_BITOPIC=False — skipping late-fusion pipeline.')
else:
    import time as _time

    # ── 6.5.1  Lexical UMAP ──────────────────────────────────────────────────
    # Run UMAP independently on the SVD-compressed lexical space.
    # Cache to disk so re-runs are instant.
    lex_umap5_path = OUTPUT_DIR / 'latefusion_lex_umap5.npy'
    lex_umap2_path = OUTPUT_DIR / 'latefusion_lex_umap2.npy'

    if lex_umap5_path.exists() and lex_umap2_path.exists():
        print('Loading cached lexical UMAP projections ...')
        lex_emb5 = np.load(lex_umap5_path)
        lex_emb2 = np.load(lex_umap2_path)
    else:
        print(f'Fitting Lexical UMAP 5D on lex_latent {lex_latent.shape} ...')
        t0 = _time.time()
        umap_lex5 = UMAP(
            n_components = 5,
            n_neighbors  = UMAP_N_NEIGHBORS,
            min_dist     = UMAP_MIN_DIST_5D,
            metric       = 'cosine',
            random_state = RANDOM_STATE,
        )
        lex_emb5 = umap_lex5.fit_transform(lex_latent).astype(np.float32)
        np.save(lex_umap5_path, lex_emb5)
        print(f'  5D done in {_time.time()-t0:.1f}s | shape: {lex_emb5.shape}')

        print('Fitting Lexical UMAP 2D for visualisation ...')
        t0 = _time.time()
        umap_lex2 = UMAP(
            n_components = 2,
            n_neighbors  = UMAP_N_NEIGHBORS,
            min_dist     = UMAP_MIN_DIST_2D,
            metric       = 'cosine',
            random_state = RANDOM_STATE,
        )
        lex_emb2 = umap_lex2.fit_transform(lex_latent).astype(np.float32)
        np.save(lex_umap2_path, lex_emb2)
        print(f'  2D done in {_time.time()-t0:.1f}s | shape: {lex_emb2.shape}')

    print(f'  lex_emb5: {lex_emb5.shape},  lex_emb2: {lex_emb2.shape}')

    # Free lex_latent — no longer needed
    del lex_latent
    gc.collect()
    print('lex_latent freed ✓')

    # ── 6.5.2  Late Fusion ───────────────────────────────────────────────────
    # Normalise each 5-D projection independently so α/β weights are on equal
    # footing, then concatenate into a 10-D fused space.
    from sklearn.preprocessing import normalize as _norm

    sem_scaled = ALPHA * _norm(bert_emb5, norm='l2')   # (N, 5)
    lex_scaled = BETA  * _norm(lex_emb5,  norm='l2')   # (N, 5)
    late_fused = np.hstack([sem_scaled, lex_scaled])   # (N, 10)

    print(f'\nLate-fused feature shape: {late_fused.shape}')
    print(f'  Semantic weight α={ALPHA}, Lexical weight β={BETA}')

    # ── 6.5.3  HDBSCAN on the 10-D late-fused space ─────────────────────────
    print(f'\nRunning Late-Fusion HDBSCAN '
          f'(min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}) ...')
    hdb_late = HDBSCAN(
        min_cluster_size       = MIN_CLUSTER_SIZE,
        min_samples            = MIN_SAMPLES,
        metric                 = 'euclidean',
        cluster_selection_method = 'eom',
        prediction_data        = True,
    )
    late_labels = hdb_late.fit_predict(late_fused)

    K_late       = len(set(late_labels)) - (1 if -1 in late_labels else 0)
    noise_late   = (late_labels == -1).sum()
    noise_pct_late = 100 * noise_late / len(late_labels)
    clustered_late = len(late_labels) - noise_late

    print(f'\n── Late-Fusion Results (α={ALPHA}, β={BETA}) ──────────────────')
    print(f'  Speeches       : {len(late_labels)}')
    print(f'  Topics found   : {K_late}')
    print(f'  Noise (-1)     : {noise_late} ({noise_pct_late:.1f}%)')
    print(f'  Clustered      : {clustered_late} ({100-noise_pct_late:.1f}%)')

    # ── 6.5.4  Head-to-head comparison ──────────────────────────────────────
    if 'bert_labels' in dir() and 'bi_labels' in dir():
        K_bert      = len(set(bert_labels)) - (1 if -1 in bert_labels else 0)
        noise_bert  = (np.array(bert_labels) == -1).sum()
        K_bi_orig   = len(set(bi_labels))   - (1 if -1 in bi_labels   else 0)
        noise_bi    = (np.array(bi_labels)   == -1).sum()

        print(f'\n── Three-way Comparison ────────────────────────────────────────')
        print(f'  {"Pipeline":<22} {"Topics":>8} {"Noise":>8} {"Clustered %":>12}')
        print(f'  {"-"*52}')
        for name, K, noise in [
            ('BERTopic baseline', K_bert,    noise_bert),
            ('BiTopic early-fuse', K_bi_orig, noise_bi),
            ('BiTopic late-fuse',  K_late,    noise_late),
        ]:
            pct = 100 * (len(late_labels) - noise) / len(late_labels)
            print(f'  {name:<22} {K:>8} {noise:>8} {pct:>11.1f}%')

    # Expose for downstream cells (keyword extraction, visualisation, etc.)
    # late_labels, hdb_late, lex_emb2 are now available globally.
    print('\nVariables exposed: late_labels, hdb_late, lex_emb2')


In [ ]:
# ── Keyword extraction for BERTopic and BiTopic ──────────────────────────────
def top_words_optimized(labels, texts, top_n=TOP_N_WORDS, stop=None):
    """Extract top terms per cluster using one global CountVectorizer."""
    if stop is None:
        stop = CUSTOM_STOPWORDS

    cv = CountVectorizer(
        stop_words=stop,
        ngram_range=CV_NGRAM_RANGE,
        min_df=CV_MIN_DF,
        token_pattern=CV_TOKEN_PATTERN,
    )
    X = cv.fit_transform(texts)
    vocab = cv.get_feature_names_out()
    labels_arr = np.array(labels)

    result = {}
    for cid in sorted(set(labels_arr)):
        mask = labels_arr == cid
        cluster_sum = np.asarray(X[mask].sum(axis=0)).ravel()
        top_idx = cluster_sum.argsort()[::-1][:top_n]
        result[cid] = vocab[top_idx].tolist()

    return result


print('Extracting keywords ...')

if RUN_BERTOPIC:
    kw_bert = top_words_optimized(bert_labels, texts)
    print(f'BERTopic: {len(kw_bert) - 1} clusters + noise')

if RUN_BITOPIC:
    kw_bi = top_words_optimized(bi_labels, texts)
    print(f'BiTopic : {len(kw_bi) - 1} clusters + noise')

print('Done ✓')


# ── Print first 10 clusters for inspection ────────────────────────────────────
def print_topic_summary(kw_dict, name, n_show=10, n_words=10):
    """Pretty-print top topics from a keyword dictionary."""
    print(f'\n── {name} — first {n_show} topics ──────────────────────────────')
    shown = 0
    for cid, words in sorted(kw_dict.items()):
        if cid == -1:
            continue
        print(f'  Topic {cid:>4}: {", ".join(words[:n_words])}')
        shown += 1
        if shown >= n_show:
            break

if RUN_BERTOPIC:
    print_topic_summary(kw_bert, 'BERTopic Baseline')

if RUN_BITOPIC:
    print_topic_summary(kw_bi, 'BiTopic')

---
## Section 9 — Noise Recovery Analysis

This section identifies speeches that were **noise in BERTopic** but were assigned
to a meaningful cluster in BiTopic — the "recovered" speeches.

In [ ]:
if not (RUN_BERTOPIC and RUN_BITOPIC):
    print('Both pipelines must run for noise recovery analysis. Skipping.')
else:
    bert_arr = np.array(bert_labels)
    bi_arr   = np.array(bi_labels)

    # Noise in BERTopic
    bert_noise_mask = bert_arr == -1
    # Of those, assigned to a real cluster in BiTopic
    recovered_mask  = bert_noise_mask & (bi_arr != -1)
    # Went from clustered → noise
    lost_mask       = (~bert_noise_mask) & (bi_arr == -1)
    # Still noise in both
    still_noise_mask = bert_noise_mask & (bi_arr == -1)

    n_recovered  = recovered_mask.sum()
    n_lost       = lost_mask.sum()
    n_still_noise = still_noise_mask.sum()
    recovery_rate = 100 * n_recovered / max(1, noise_bert)

    print('════════════════════════════════════════════════════════════════')
    print('  Noise Recovery Analysis')
    print('════════════════════════════════════════════════════════════════')
    print(f'  BERTopic noise (total)     : {noise_bert}')
    print(f'  Recovered by BiTopic       : {n_recovered} ({recovery_rate:.1f}% of BERTopic noise)')
    print(f'  Still noise in BiTopic     : {n_still_noise}')
    print(f'  Lost to noise by BiTopic   : {n_lost}  (was clustered in BERTopic)')
    print()

    if n_recovered > 0:
        # Distribution of recovered speeches across BiTopic clusters
        recovered_topics = bi_arr[recovered_mask]
        topic_counts = pd.Series(recovered_topics).value_counts().sort_index()
        print(f'  Recovered speeches spread across {len(topic_counts)} BiTopic topics:')
        for tid, cnt in topic_counts.items():
            kw_preview = ', '.join(kw_bi.get(tid, [])[:6])
            print(f'    Topic {tid:>4}: {cnt:>4} speeches | {kw_preview}')
    else:
        print('  No speeches recovered from noise.')

In [ ]:
if not (RUN_BERTOPIC and RUN_BITOPIC):
    print('Skipping.')
else:
    # ── Print example recovered speeches ─────────────────────────────────────
    recovered_idx = np.where(recovered_mask)[0]
    n_show        = min(RECOVERY_EXAMPLE_N, len(recovered_idx))

    print(f'\n── {n_show} Example Recovered Speeches ──────────────────────────────')
    for i, idx in enumerate(recovered_idx[:n_show]):
        assigned_topic = bi_arr[idx]
        kw_preview     = ', '.join(kw_bi.get(assigned_topic, [])[:8])
        speech_preview = texts[idx][:300].replace('\n', ' ')
        speaker_info   = df.get('speaker', pd.Series(['?']*len(df))).iloc[idx]
        date_info      = df.get('date', pd.Series(['?']*len(df))).iloc[idx]
        print(f'\n  [{i+1}] Speech index {idx}')
        print(f'      Speaker : {speaker_info}')
        print(f'      Date    : {date_info}')
        print(f'      BiTopic : Topic {assigned_topic} | {kw_preview}')
        print(f'      Text    : {speech_preview}...')
        print(f'      {"-"*70}')

---
## Section 10 — Visualisation

In [ ]:
# ── Helper: make readable topic label from keyword dict ───────────────────────
def make_label(cid, kw_dict, n=5):
    if cid == -1:
        return 'Noise'
    words = kw_dict.get(cid, [])
    return f'T{cid}: {", ".join(words[:n])}'

In [ ]:
# ── 10.1  Topic count & noise comparison bar chart ────────────────────────────
if RUN_BERTOPIC and RUN_BITOPIC:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Topic count
    ax = axes[0]
    bars = ax.bar(['BERTopic', 'BiTopic'], [K_bert, K_bi],
                  color=['steelblue', 'coral'], width=0.5, edgecolor='white')
    ax.bar_label(bars, fmt='%d', padding=3, fontsize=12)
    ax.set_title('Number of Topics Found', fontweight='bold')
    ax.set_ylabel('Topics (K)')
    ax.set_ylim(0, max(K_bert, K_bi) * 1.2)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    # Noise percentage
    ax = axes[1]
    bars2 = ax.bar(['BERTopic', 'BiTopic'], [noise_pct_b, noise_pct_bi],
                   color=['steelblue', 'coral'], width=0.5, edgecolor='white')
    ax.bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=12)
    ax.set_title('Noise Percentage', fontweight='bold')
    ax.set_ylabel('Noise (%)')
    ax.set_ylim(0, max(noise_pct_b, noise_pct_bi) * 1.3)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    plt.suptitle('BERTopic vs BiTopic — Summary Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'comparison_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → comparison_summary.png')

In [ ]:
# ── 10.2  2D Scatter plots: BERTopic vs BiTopic ───────────────────────────────
def scatter_clusters(emb2, labels, kw_dict, title, ax, max_labels=15, sample_n=10000):
    """
    Plot 2D UMAP scatter with per-cluster colours.
    Large corpora are subsampled for speed; all noise points plotted in grey.
    """
    labels_arr  = np.array(labels)
    unique_cids = sorted(set(labels_arr))

    # Subsample if very large
    if len(labels_arr) > sample_n:
        rng  = np.random.default_rng(RANDOM_STATE)
        idx  = rng.choice(len(labels_arr), size=sample_n, replace=False)
        emb2      = emb2[idx]
        labels_arr = labels_arr[idx]
        unique_cids = sorted(set(labels_arr))

    cmap    = plt.get_cmap('tab20', max(1, len(unique_cids) - 1))
    color_i = 0

    for cid in unique_cids:
        mask = labels_arr == cid
        if cid == -1:
            ax.scatter(emb2[mask, 0], emb2[mask, 1],
                       c='lightgrey', s=4, alpha=0.3, label='Noise', rasterized=True)
        else:
            lbl = make_label(cid, kw_dict) if color_i < max_labels else None
            ax.scatter(emb2[mask, 0], emb2[mask, 1],
                       c=[cmap(color_i % 20)], s=6, alpha=0.6,
                       label=lbl, rasterized=True)
            color_i += 1

    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('UMAP-1', fontsize=8)
    ax.set_ylabel('UMAP-2', fontsize=8)
    ax.tick_params(labelsize=7)

    if color_i <= max_labels:
        ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1),
                  fontsize=6, markerscale=2, frameon=True)


n_cols = (1 if RUN_BERTOPIC else 0) + (1 if RUN_BITOPIC else 0)
if n_cols > 0:
    fig, axes = plt.subplots(1, n_cols, figsize=(9 * n_cols, 7))
    if n_cols == 1:
        axes = [axes]

    col = 0
    if RUN_BERTOPIC:
        scatter_clusters(bert_emb2, bert_labels, kw_bert,
                         f'BERTopic Baseline\nK={K_bert}, Noise={noise_pct_b:.1f}%',
                         axes[col])
        col += 1

    if RUN_BITOPIC:
        scatter_clusters(bi_emb2, bi_labels, kw_bi,
                         f'BiTopic (α={ALPHA}, β={BETA})\nK={K_bi}, Noise={noise_pct_bi:.1f}%',
                         axes[col])

    plt.suptitle('2D UMAP Cluster Projections', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'scatter_clusters.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → scatter_clusters.png')

In [ ]:
# ── 10.3  Topic size distribution ─────────────────────────────────────────────
def topic_size_series(labels):
    """Return a Series of cluster sizes (noise excluded), sorted descending."""
    labels_arr = np.array(labels)
    sizes = pd.Series(labels_arr[labels_arr != -1]).value_counts().sort_values(ascending=False)
    return sizes


if RUN_BERTOPIC and RUN_BITOPIC:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, (name, labels) in zip(axes, [
        ('BERTopic', bert_labels),
        (f'BiTopic α={ALPHA}', bi_labels)
    ]):
        sizes = topic_size_series(labels)
        ax.bar(range(len(sizes)), sizes.values, color='steelblue', width=0.8)
        ax.set_title(f'{name} — Topic Sizes (top {len(sizes)} clusters)', fontweight='bold')
        ax.set_xlabel('Cluster rank')
        ax.set_ylabel('# speeches')
        ax.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'topic_sizes.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → topic_sizes.png')

In [ ]:
# ── 10.4  Noise recovery stacked bar ──────────────────────────────────────────
if RUN_BERTOPIC and RUN_BITOPIC:
    categories   = ['Recovered\n(BERTopic noise\n→ BiTopic cluster)',
                    'Permanent noise\n(noise in both)',
                    'Lost\n(BERTopic cluster\n→ BiTopic noise)']
    values       = [n_recovered, n_still_noise, n_lost]
    colours      = ['#4caf50', '#9e9e9e', '#f44336']

    fig, ax = plt.subplots(figsize=(8, 5))
    bars    = ax.bar(categories, values, color=colours, width=0.5, edgecolor='white')
    ax.bar_label(bars, fmt='%d', padding=3, fontsize=12)
    ax.set_title(f'Noise Recovery: BERTopic → BiTopic (α={ALPHA}, β={BETA})',
                 fontweight='bold')
    ax.set_ylabel('# speeches')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'noise_recovery.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → noise_recovery.png')

In [ ]:
# ── 10.5  Temporal topic evolution ────────────────────────────────────────────
def temporal_line_chart(labels, years, kw_dict, title, ax, top_n=20):
    """
    % share of each topic per year (top_n topics by total count).
    """
    df_t  = pd.DataFrame({'cluster': labels, 'year': years})
    df_t  = df_t[df_t['cluster'] != -1]
    pivot = df_t.groupby(['cluster', 'year']).size().unstack(fill_value=0)
    year_totals = df_t.groupby('year').size()
    pct   = pivot.div(year_totals, axis=1) * 100

    top_clusters = pivot.sum(axis=1).nlargest(top_n).index
    pct_plot     = pct.loc[top_clusters]

    cmap   = plt.get_cmap('tab20', len(pct_plot))
    x_vals = sorted(pct.columns)

    for i, (cid, row) in enumerate(pct_plot.iterrows()):
        label = make_label(cid, kw_dict, n=4)
        ax.plot(x_vals, [row.get(yr, 0) for yr in x_vals],
                marker='o', linewidth=1.8, markersize=4,
                color=cmap(i), label=label)

    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Year', fontsize=8)
    ax.set_ylabel('% of yearly speeches', fontsize=8)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1),
              fontsize=6, ncol=1, frameon=True)


if RUN_BERTOPIC and RUN_BITOPIC:
    TOP_N_TEMPORAL = min(15, min(K_bert, K_bi))
    fig, axes      = plt.subplots(2, 1, figsize=(16, 12))

    temporal_line_chart(bert_labels, years_arr, kw_bert,
                        f'BERTopic — Topic Share Over Time (top {TOP_N_TEMPORAL})',
                        axes[0], top_n=TOP_N_TEMPORAL)
    temporal_line_chart(bi_labels, years_arr, kw_bi,
                        f'BiTopic (α={ALPHA}) — Topic Share Over Time (top {TOP_N_TEMPORAL})',
                        axes[1], top_n=TOP_N_TEMPORAL)

    plt.suptitle('Temporal Topic Evolution', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'temporal_topics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → temporal_topics.png')

---
## Section 11 — Save Outputs

In [ ]:
# ── 11.1  Cluster assignment CSV ──────────────────────────────────────────────
base_cols = ['date', 'year', 'speaker'] if 'speaker' in df.columns else ['date', 'year']
if 'speech_id' in df.columns:
    base_cols = ['speech_id'] + base_cols

df_out = df[base_cols].copy()

if RUN_BERTOPIC:
    df_out['bertopic_cluster'] = bert_labels

if RUN_BITOPIC:
    df_out['bitopic_cluster']  = bi_labels

if RUN_BERTOPIC and RUN_BITOPIC:
    df_out['recovered_from_noise'] = recovered_mask.astype(int)
    df_out['lost_to_noise']        = lost_mask.astype(int)

out_csv = OUTPUT_DIR / 'v13_cluster_assignments.csv'
df_out.to_csv(out_csv, index=False)
print(f'Saved → {out_csv}')

In [ ]:
# ── 11.2  Comparison table CSV ────────────────────────────────────────────────
if RUN_BERTOPIC or RUN_BITOPIC:
    rows = []
    N    = len(df)

    if RUN_BERTOPIC:
        bert_arr  = np.array(bert_labels)
        k_bert    = len(set(bert_arr)) - (1 if -1 in bert_arr else 0)
        noise_bert = int((bert_arr == -1).sum())
        rows.append({
            'pipeline'      : 'BERTopic (semantic only)',
            'n_topics'      : k_bert,
            'noise_count'   : noise_bert,
            'noise_pct'     : round(100 * noise_bert / N, 2),
            'clustered_count': N - noise_bert,
            'clustered_pct' : round(100 * (N - noise_bert) / N, 2),
        })

    if RUN_BITOPIC:
        bi_arr    = np.array(bi_labels)
        k_bi      = len(set(bi_arr)) - (1 if -1 in bi_arr else 0)
        noise_bi  = int((bi_arr == -1).sum())
        rows.append({
            'pipeline'      : 'BiTopic (early fusion)',
            'n_topics'      : k_bi,
            'noise_count'   : noise_bi,
            'noise_pct'     : round(100 * noise_bi / N, 2),
            'clustered_count': N - noise_bi,
            'clustered_pct' : round(100 * (N - noise_bi) / N, 2),
        })

    # Include late-fusion results if the cell ran successfully
    if 'late_labels' in dir():
        late_arr   = np.array(late_labels)
        k_late     = len(set(late_arr)) - (1 if -1 in late_arr else 0)
        noise_late = int((late_arr == -1).sum())
        rows.append({
            'pipeline'      : 'BiTopic (late fusion)',
            'n_topics'      : k_late,
            'noise_count'   : noise_late,
            'noise_pct'     : round(100 * noise_late / N, 2),
            'clustered_count': N - noise_late,
            'clustered_pct' : round(100 * (N - noise_late) / N, 2),
        })

    df_compare   = pd.DataFrame(rows)
    compare_path = OUTPUT_DIR / 'v13_comparison.csv'
    df_compare.to_csv(compare_path, index=False)
    print(f'Saved → {compare_path}')
    print()
    print(df_compare.to_string(index=False))


In [ ]:
# ── 11.3  Keywords JSON ───────────────────────────────────────────────────────
kw_out = {}
if RUN_BERTOPIC:
    kw_out['bertopic'] = {str(k): v for k, v in kw_bert.items()}
if RUN_BITOPIC:
    kw_out['bitopic']  = {str(k): v for k, v in kw_bi.items()}

kw_path = OUTPUT_DIR / 'v13_keywords.json'
with open(kw_path, 'w', encoding='utf-8') as f:
    json.dump(kw_out, f, indent=2, ensure_ascii=False)
print(f'Saved → {kw_path}')

In [ ]:
# ── 11.4  Recovered speeches export ──────────────────────────────────────────
if RUN_BERTOPIC and RUN_BITOPIC and n_recovered > 0:
    df_recovered = df.iloc[recovered_idx].copy()
    df_recovered['bitopic_cluster'] = bi_arr[recovered_idx]
    df_recovered['bitopic_keywords'] = [
        ', '.join(kw_bi.get(c, [])[:10])
        for c in df_recovered['bitopic_cluster']
    ]
    rec_path = OUTPUT_DIR / 'v13_recovered_speeches.csv'
    df_recovered.to_csv(rec_path, index=True)
    print(f'Saved → {rec_path}  ({len(df_recovered)} recovered speeches)')

In [ ]:
# ── 11.5  LLM macro-categorisation prompt (BiTopic) ──────────────────────────
if RUN_BITOPIC:
    llm_path = OUTPUT_DIR / 'llm_bitopic_prompt.txt'
    with open(llm_path, 'w', encoding='utf-8') as f:
        f.write(
            "I am providing a list of micro-topics extracted from Sri Lankan parliamentary "
            "debates using a BiTopic pipeline (semantic + lexical fusion), represented by "
            "their top keywords. Please act as a Political Data Scientist. Read these "
            "micro-topics and group them into exactly 10–15 overarching 'Macro-Categories' "
            "(e.g., Economy & IMF, Healthcare & Covid, National Security, etc.).\n\n"
            "Output ONLY a clean Python dictionary where the keys are the Macro-Category "
            "names (strings) and the values are lists of integers (Micro-Topic IDs). "
            "Do not include any other explanations.\n\n"
            "--- BITOPIC MICRO TOPICS ---\n"
        )
        for cid, words in sorted(kw_bi.items()):
            if cid == -1:
                continue
            f.write(f'Topic {cid}: {", ".join(words[:10])}\n')
    print(f'Saved → {llm_path}')
    print('Open this file, copy ALL the text, and paste it into Claude!')

print('\n✓ All outputs saved.')

---
## Section 12 — Summary & Next Steps

Run this cell last for a full readable summary.

In [ ]:
print('═' * 65)
print('  v13 BiTopic — Final Summary')
print('═' * 65)
print(f'  Corpus              : {len(df)} speeches (2017–2026)')
print(f'  Embedding model     : {EMBEDDING_MODEL}')
print(f'  UMAP n_neighbors    : {UMAP_N_NEIGHBORS}')
print(f'  HDBSCAN mcs/ms      : {MIN_CLUSTER_SIZE}/{MIN_SAMPLES}')
print()

if RUN_BERTOPIC:
    print(f'  BERTopic (baseline)')
    print(f'    Topics (K)        : {K_bert}')
    print(f'    Noise             : {noise_bert} ({noise_pct_b:.1f}%)')
    print(f'    Silhouette        : {sil_bert:.4f}')

if RUN_BITOPIC:
    print(f'\n  BiTopic (α={ALPHA}, β={BETA})')
    print(f'    Topics (K)        : {K_bi}')
    print(f'    Noise             : {noise_bi} ({noise_pct_bi:.1f}%)')
    print(f'    Silhouette        : {sil_bi:.4f}')

if RUN_BERTOPIC and RUN_BITOPIC:
    print(f'\n  Noise recovery      : {n_recovered} speeches ({recovery_rate:.1f}% of BERTopic noise)')
    print(f'  ΔTopics             : {K_bi - K_bert:+d}')
    print(f'  ΔNoise              : {noise_bi - noise_bert:+d}')

print(f'\n  Output dir          : {OUTPUT_DIR}')
print('═' * 65)
print()
print('Next steps:')
print('  1. Open llm_bitopic_prompt.txt and paste into Claude for macro-categorisation.')
print('  2. Tune ALPHA/BETA in Section 2 and re-run Section 6 for different fusion weights.')
print('  3. For a v14, consider: soft cluster membership, BERTopic soft-cluster noise assignment,')
print('     or adding a TF-IDF layer on top of the BiTopic clusters.')